# Gate 실험: Go가 채점한 case로 하는 임계값 조사

조사 질문: *묻지 않고 action을 실행하려면 어떤 confidence 임계값이 유망한가?*

run(`pnpm eval replay` 또는 `run`)의 `cases.jsonl`을 읽고, Go가 채점한 `quality.outcome`을 쓴다.

In [ ]:
from pathlib import Path

from snapdone_eval_lab import analysis
from snapdone_eval_lab.artifacts import load_run, variant_costs
from snapdone_eval_lab.datasets import repository_root
from snapdone_eval_lab.plots import bar, sweep_plot

ROOT = repository_root()
RUN_DIR = ROOT / "tools/evals/results/lab-python"   # 직접 만든 run. 예: 02_prompt_model_lab에서 내보낸 것
if not RUN_DIR.exists():
    RUN_DIR = ROOT / "apps/api/internal/evaluation/testdata/artifacts/replay-golden"  # Go 자신의 golden run, 3 case
    print("results에 run이 없어 Go의 golden 산출물을 대신 읽는다:", RUN_DIR)

run = load_run(RUN_DIR)
print(run.run_id, "·", run.metadata["mode"], "· 정책", run.policy, "· 공식 채점 대상:", run.summary["officialEligible"])
cases = run.cases
cases[["case_id", "variant_id", "status", "outcome", "confidence", "predicted_action", "critical"]]

In [ ]:
# confidence 칸. precision = Go 통과 수 / 칸의 case 수, coverage = 칸 / 전체 case
buckets = analysis.confidence_buckets(cases)
bar(buckets["cases"], "cases per confidence level")
buckets

In [ ]:
# confidence가 각 임계값 이상이면 자동 실행. 나머지는 넘긴다.
sweep = analysis.threshold_sweep(cases)
sweep_plot(sweep)
sweep

In [ ]:
# Go가 failed로 채점한 high confidence 답: 임계값이 >= high면 묻지 않고 실행했을 case
analysis.high_confidence_wrong(cases)

In [ ]:
# Go summary의 variant별 지연 시간과 토큰. replay run에는 없고 note가 이유를 말한다.
variant_costs(run.summary)

## 후보 올리기

후보를 공식으로 시험하려면:

1. **Go의 기존 실험 설정으로 동작을 적는다.** confidence 임계값은 계단식의 `escalateOn` 목록이다: *>= high면 자동 실행, 그 아래는 묻거나 넘김* 후보는 `{"cascade": {"escalateOn": ["low", "medium"]}}`이고, 이것이 `tools/evals/experiments/cascade.json`이다. 다른 후보는 그 옆에 새 파일로 두고 `--variant <setting>@<provider>:<model>,<escalate model>`로 돌린다.
2. **한계를 gate 정책 파일로 적는다.** `tools/evals/gates/` 아래에 Go의 `GatePolicy`가 읽는 모양으로: `{"version": "<name>-v1", "maxNewCriticalErrors": 0, "maxPassRateDropPp": 2}`. 적은 규칙만 판정한다.
3. **Go가 정하게 한다:**

```bash
pnpm eval run --dataset <dataset> --variant <baseline> --variant <candidate> --allow-api --max-api-calls N --run-id <run>
pnpm eval compare --baseline <run>:<baseline> --candidate <run>:<candidate> --gate tools/evals/gates/<name>.json
```

종료 코드 4는 gate 실패이고, 판정은 `results/comparisons/<id>/comparison.json`에 있다.